In [1]:
#!sudo /bin/bash -c "(source /venv/bin/activate; pip install --upgrade google-api-python-client)"

     |████████████████████████████████| 11.9 MB 12.3 MB/s eta 0:00:01
     |████████████████████████████████| 139 kB 42.3 MB/s eta 0:00:01
     |████████████████████████████████| 96 kB 8.2 MB/s  eta 0:00:01
     |████████████████████████████████| 50 kB 7.2 MB/s  eta 0:00:01
     |████████████████████████████████| 220 kB 26.2 MB/s eta 0:00:01
     |████████████████████████████████| 309 kB 59.6 MB/s eta 0:00:01


In [15]:
import logging
import os

import numpy as np

import ck_marketing.hunterio.hunterapi as cmahuhun
import ck_marketing.linkedin.profile_filtering as cmliprfi
import helpers.hgoogle_file_api as hgfiapi
from ck_marketing.hunterio.hunterapi import GoogleSheetsHelper, HunterIO
from ck_marketing.linkedin.phantombuster_api import Phantom

## Extracting Profiles

In [5]:
# Get the API keys from the environment variables.
phantom_api_key = os.getenv("Phantom_API_KEY")
hunter_api_key = os.getenv("Hunter_API_KEY")

In [6]:
# Initialize the Phantom instance.
phantom = Phantom(phantom_api_key)

In [7]:
# Get and print all agents and their IDs.
agents = phantom.get_all_agents()
print("List of all agents and their IDs:\n")
for agent in agents:
    print(f"Agent Name: {agent['name']}, Agent ID: {agent['id']}")

List of all agents and their IDs:

Agent Name: Bloomberg, Agent ID: 1809505480671122
Agent Name: Flutter_decision_makers.export_search, Agent ID: 2471526793293821
Agent Name: Untitled LinkedIn Connections Export, Agent ID: 795029239791557


In [8]:
# Get agent ID and name.
AGENT_ID = "1809505480671122"

In [9]:
specific_agent_name = phantom.get_agent_name(AGENT_ID)
print(f"Selected Phantom: {specific_agent_name}")

Selected Phantom: Bloomberg


In [21]:
# Google Drive Setup.
google_creds_path = "service.json"
google_sheet_helper = GoogleSheetsHelper(google_creds_path)
drive_folder_id = "1fpsUZ8Nd52FGKxOuqVf11hzEetwvVoDq"
sheet_name = f"{specific_agent_name}_search_export"
tab_name = "search_export"

In [25]:
# Launch the agent and get the results in a DataFrame
phantom.launch_agent(AGENT_ID)
result_response_json = phantom.fetch_agent_results(AGENT_ID)
csv_url = phantom.get_csv_url(result_response_json.get("output", ""))
df = phantom.download_csv(csv_url)
df = df.replace([np.nan, np.inf, -np.inf], "", inplace=False)
print("DataFrame is fetched")
# print(df.head(2))

DataFrame is fetched


In [29]:
# Create the Google Sheet and get the file ID.
file_id = hgfiapi.create_empty_google_file("sheet", sheet_name, drive_folder_id)

if file_id:
    # Initialize Google Sheets Helper.
    google_sheets_helper = GoogleSheetsHelper(google_creds_path)

    # Open the Google Sheet by file ID.
    sheet = google_sheets_helper.google_account.open_by_key(file_id)

    # Access the default tab (the first worksheet).
    default_worksheet = sheet.get_worksheet(0)

    # Rename the default tab to the desired name.
    new_tab_name = "search_export"
    default_worksheet.update_title(new_tab_name)

    # Write the DataFrame to the renamed default tab.
    google_sheets_helper.write_results(file_id, df, new_tab_name)

    print(
        f"DataFrame written to Google Sheet '{new_tab_name}' in file ID '{file_id}' successfully."
    )
else:
    print("Failed to create the Google Sheet.")

INFO:helpers.hgoogle_file_api:Created a new Google sheet 'Bloomberg_search_export'.
/venv/lib/python3.9/site-packages/gspread/worksheet.py:1069: UserWarning: [Deprecated][in version 6.0.0]: method signature will change to: 'Worksheet.update(value = [[]], range_name=)' arguments 'range_name' and 'values' will swap, values will be mandatory of type: 'list(list(...))'
  warnings.warn(
INFO:ck_marketing.hunterio.hunterapi:Email extraction completed. Results saved in the new tab: search_export


DataFrame written to Google Sheet 'search_export' in file ID '1Etx_Ee9WihmgKAbn4PDN2JvAGWKFLydBAIdjYkuQRwY' successfully.


## Clean Profiles

In [23]:
# df = google_sheet_helper.read_sheet('1Etx_Ee9WihmgKAbn4PDN2JvAGWKFLydBAIdjYkuQRwY')
# file_id = '1Etx_Ee9WihmgKAbn4PDN2JvAGWKFLydBAIdjYkuQRwY'
sheet = google_sheet_helper.google_account.open_by_key(file_id)

In [14]:
words = [
    "sales",
    "research",
    "delivery",
    "income",
    "television",
    "talent",
    "climate",
    "voice",
    "media",
]

In [16]:
filtered_df = cmliprfi.filter_df(df, "title", words, "remove")

INFO:ck_marketing.linkedin.profile_filtering:Filtered dataframe to remove rows where 'title' contains any of ['sales', 'research', 'delivery', 'income', 'television', 'talent', 'climate', 'voice', 'media'].
INFO:ck_marketing.linkedin.profile_filtering:135 entries were removed.
INFO:ck_marketing.linkedin.profile_filtering:Entries before filter: 642, Entries after filter: 507
INFO:ck_marketing.linkedin.profile_filtering:Original entries: 642
INFO:ck_marketing.linkedin.profile_filtering:Remaining entries after filtering: 507
INFO:ck_marketing.linkedin.profile_filtering:Removed entries: 135
INFO:ck_marketing.linkedin.profile_filtering:Percentage of entries removed: 21.03%


In [26]:
if file_id:
    # Create a new tab called 'cleaned_profiles'.
    cleaned_profiles_tab = sheet.add_worksheet(
        title="cleaned_profiles", rows="100", cols="20"
    )

    # Write the filtered DataFrame to the new tab.
    google_sheet_helper.write_results(file_id, filtered_df, "cleaned_profiles")

    print(
        f"Filtered DataFrame written to new tab 'cleaned_profiles' in Google Sheet with file ID '{file_id}' successfully."
    )
else:
    print("Failed to create the Google Sheet.")

/venv/lib/python3.9/site-packages/gspread/worksheet.py:1069: UserWarning: [Deprecated][in version 6.0.0]: method signature will change to: 'Worksheet.update(value = [[]], range_name=)' arguments 'range_name' and 'values' will swap, values will be mandatory of type: 'list(list(...))'
  warnings.warn(
INFO:ck_marketing.hunterio.hunterapi:Email extraction completed. Results saved in the new tab: cleaned_profiles


Filtered DataFrame written to new tab 'cleaned_profiles' in Google Sheet with file ID '1Etx_Ee9WihmgKAbn4PDN2JvAGWKFLydBAIdjYkuQRwY' successfully.


## Extracting emails using hunterio

In [28]:
# Set up logging to see print statements and warnings.
logging.basicConfig(level=logging.INFO)

first_name_col = "firstName"
last_name_col = "lastName"
company_col = "companyName"
tab_name = "cleaned_profiles"

cmahuhun.process_records(
    api_key=hunter_api_key,
    google_creds_path=google_creds_path,
    file_id=file_id,
    first_name_col=first_name_col,
    last_name_col=last_name_col,
    company_col=company_col,
    tab_name=tab_name,
)

INFO:ck_marketing.hunterio.hunterapi:Starting process to read records and find emails
INFO:ck_marketing.hunterio.hunterapi:Finding bulk emails using company name
INFO:ck_marketing.hunterio.hunterapi:Writing results to Google Sheets
/venv/lib/python3.9/site-packages/gspread/worksheet.py:1069: UserWarning: [Deprecated][in version 6.0.0]: method signature will change to: 'Worksheet.update(value = [[]], range_name=)' arguments 'range_name' and 'values' will swap, values will be mandatory of type: 'list(list(...))'
  warnings.warn(
INFO:ck_marketing.hunterio.hunterapi:Email extraction completed. Results saved in the new tab: hunter_results
INFO:ck_marketing.hunterio.hunterapi:Total records processed: 507
INFO:ck_marketing.hunterio.hunterapi:Emails found: 23
INFO:ck_marketing.hunterio.hunterapi:Emails not found: 484
INFO:ck_marketing.hunterio.hunterapi:Number of unique companies: 24
INFO:ck_marketing.hunterio.hunterapi:Percentage of emails found: 4.54%


INFO:ck_marketing.hunterio.hunterapi:Companies by number of records:
INFO:ck_marketing.hunterio.hunterapi:companyName
Bloomberg LP                    270
Bloomberg                       182
BloombergNEF                     20
Bloomberg Television              5
Bloomberg L.P.                    4
彭博资讯                              4
Bloomberg New Energy Finance      2
New Energy Finance                2
Bloomberg TV                      2
Bloomberg News                    2
Bloomberg PolarLake               1
Bloomberg NEF                     1
The AES Corporation               1
Bloomberg Inc                     1
PalAmerican Security              1
Bloomberg Businessweek            1
Victor Brown Associates           1
Seven Saints Productions          1
Self-employed                     1
Bloomberg Markets                 1
彭博資訊                              1
Bloomberg.com                     1
Bloomberg LLP                     1
The Carlyle Group                 1
Name: count, dtype

## Dropcontact for remaining emails

### Waiting for API keys

## Email Verification

In [36]:
# df_drop = google_sheet_helper.read_sheet("1Etx_Ee9WihmgKAbn4PDN2JvAGWKFLydBAIdjYkuQRwY", "hunter_results")
# file_id = '1Etx_Ee9WihmgKAbn4PDN2JvAGWKFLydBAIdjYkuQRwY'
# sheet = google_sheet_helper.google_account.open_by_key(file_id)

In [38]:
hunter_instance = HunterIO(hunter_api_key)
verified_df = hunter_instance.verify_emails(df_drop, "hunter_extracted_email")

In [41]:
if file_id:
    # Create a new tab.
    cleaned_profiles_tab = sheet.add_worksheet(
        title="hunter_verification", rows="100", cols="20"
    )

    # Write the filtered DataFrame to the new tab.
    google_sheet_helper.write_results(file_id, verified_df, "hunter_verification")

    print(
        f"DataFrame written to new tab 'hunter_verification' in Google Sheet with file ID '{file_id}' successfully."
    )
else:
    print("Failed to create the Google Sheet.")

INFO:ck_marketing.hunterio.hunterapi:Email extraction completed. Results saved in the new tab: hunter_verification


DataFrame written to new tab 'hunter_verification' in Google Sheet with file ID '1Etx_Ee9WihmgKAbn4PDN2JvAGWKFLydBAIdjYkuQRwY' successfully.


## Final dataframe

In [48]:
final_df = google_sheet_helper.read_sheet(
    "1Etx_Ee9WihmgKAbn4PDN2JvAGWKFLydBAIdjYkuQRwY", "hunter_verification"
)

In [49]:
final_df = final_df[
    [
        "fullName",
        "profileUrl",
        "title",
        "hunter_extracted_email",
        "hunter_verification",
    ]
]
# Step 2: Filter out rows where 'hunter_extracted_email' is empty.
final_df = final_df[
    final_df["hunter_extracted_email"].notna()
    & (final_df["hunter_extracted_email"] != "")
]

In [50]:
if file_id:
    # Create a new tab.
    cleaned_profiles_tab = sheet.add_worksheet(
        title="final_df", rows="100", cols="20"
    )
    # Write the filtered DataFrame to the new tab.
    google_sheet_helper.write_results(file_id, final_df, "final_df")
    print(
        f"DataFrame written to new tab 'hunter_verification' in Google Sheet with file ID '{file_id}' successfully."
    )
else:
    print("Failed to create the Google Sheet.")

/venv/lib/python3.9/site-packages/gspread/worksheet.py:1069: UserWarning: [Deprecated][in version 6.0.0]: method signature will change to: 'Worksheet.update(value = [[]], range_name=)' arguments 'range_name' and 'values' will swap, values will be mandatory of type: 'list(list(...))'
  warnings.warn(
INFO:ck_marketing.hunterio.hunterapi:Email extraction completed. Results saved in the new tab: final_df


DataFrame written to new tab 'hunter_verification' in Google Sheet with file ID '1Etx_Ee9WihmgKAbn4PDN2JvAGWKFLydBAIdjYkuQRwY' successfully.
